In [1]:
from typing import Literal, Optional

import instructor

from google.genai import Client
from instructor import Mode
from instructor.v2 import from_genai
from pydantic import BaseModel, Field


PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "global"
MODEL_ID = "gemini-3.1-flash-lite"

In [2]:
raw_client = Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

client = from_genai(
    raw_client,
    mode=Mode.JSON,
)

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model: {MODEL_ID}")

Project: leafy-guide-497515-m4
Location: global
Model: gemini-3.1-flash-lite


In [3]:
def call_api(
    messages,
    response_model,
    model=MODEL_ID,
    max_retries=2,
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        response_model=response_model,
        max_retries=max_retries,
    )

    return response

In [4]:
TicketCategory = Literal[
    "Account",
    "Billing",
    "Bug",
    "Performance",
    "Feature Request",
    "Security",
]

In [5]:
TicketCategory = Literal[
    "Account",
    "Billing",
    "Bug",
    "Performance",
    "Feature Request",
    "Security",
]

In [6]:
TicketPriority = Literal[
    "Low",
    "Medium",
    "High",
    "Critical",
]

In [7]:
class SupportTicket(BaseModel):
    category: TicketCategory = Field(
        description=(
            "The primary category of the support ticket. "
            "Select exactly one value allowed by the schema."
        )
    )

    priority: TicketPriority = Field(
        description=(
            "The urgency of the support ticket. "
            "Use Critical for security incidents, data loss, "
            "or complete service outages. "
            "Use High when an important feature is unavailable. "
            "Use Medium when the problem disrupts work but has a workaround. "
            "Use Low for questions, suggestions, and minor inconveniences."
        )
    )

    requires_immediate_action: bool = Field(
        description=(
            "Whether the ticket requires immediate intervention. "
            "Return true for critical security incidents, data loss, "
            "or complete service outages. Otherwise return false."
        )
    )

    summary: str = Field(
        min_length=5,
        max_length=160,
        description=(
            "A concise summary of the reported problem. "
            "Do not add information that is not present in the ticket."
        )
    )

In [8]:
system_prompt = """
Classify the provided customer support ticket.

Use only values allowed by the response schema.

Category rules:
- Account: login, password, profile, or account access problems.
- Billing: payments, invoices, refunds, subscriptions, or duplicate charges.
- Bug: software behaviour that is incorrect or causes an error.
- Performance: slow loading, timeouts, or excessive resource usage.
- Feature Request: requests for new functionality or improvements.
- Security: suspicious access, exposed data, compromised credentials, or vulnerabilities.

Priority rules:
- Critical: security incident, data loss, or complete service outage.
- High: an important feature is completely unavailable.
- Medium: the issue disrupts work but a workaround may exist.
- Low: a question, suggestion, or minor inconvenience.

Do not invent information that is not explicitly present in the ticket.
Return a concise summary written in English.
"""

In [9]:
test_tickets = [
    "I forgot my password and cannot access my account.",
    "I was charged twice for the same monthly subscription.",
    "The application crashes whenever I upload a PDF file.",
    "The analytics dashboard takes more than thirty seconds to load.",
    "Please add an option to export reports to Excel.",
    "Someone logged into my account from an unknown device.",
]

In [10]:
results = []

for ticket in test_tickets:
    response = call_api(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": ticket,
            },
        ],
        response_model=SupportTicket,
        max_retries=2,
    )

    result = {
        "ticket": ticket,
        **response.model_dump(),
    }

    results.append(result)

    print(f"Ticket: {ticket}")
    print(response)
    print("=" * 100)

Ticket: I forgot my password and cannot access my account.
category='Account' priority='Medium' requires_immediate_action=False summary='User is unable to access their account due to a forgotten password.'
Ticket: I was charged twice for the same monthly subscription.
category='Billing' priority='Medium' requires_immediate_action=False summary='Customer reported a duplicate charge for their monthly subscription.'
Ticket: The application crashes whenever I upload a PDF file.
category='Bug' priority='High' requires_immediate_action=False summary='The application crashes when attempting to upload a PDF file.'
Ticket: The analytics dashboard takes more than thirty seconds to load.
category='Performance' priority='Medium' requires_immediate_action=False summary='The analytics dashboard is experiencing slow loading times exceeding thirty seconds.'
Ticket: Please add an option to export reports to Excel.
category='Feature Request' priority='Low' requires_immediate_action=False summary='Reques

In [12]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df

,ticket,category,priority,requires_immediate_action,summary
0,I forgot my password and cannot access my acco...,Account,Medium,False,User is unable to access their account due to ...
1,I was charged twice for the same monthly subsc...,Billing,Medium,False,Customer reported a duplicate charge for their...
2,The application crashes whenever I upload a PD...,Bug,High,False,The application crashes when attempting to upl...
3,The analytics dashboard takes more than thirty...,Performance,Medium,False,The analytics dashboard is experiencing slow l...
4,Please add an option to export reports to Excel.,Feature Request,Low,False,Request to add functionality to export reports...
5,Someone logged into my account from an unknown...,Security,Critical,True,Unauthorized account access detected from an u...
